In [20]:
!pip install datasets sentence-transformers faiss-cpu transformers

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [21]:
import re
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


In [22]:
#config
DATA_PATH = "/kaggle/input/wikipedia/AllCombined.txt"

CHUNK_TOKENS = 400
CHUNK_OVERLAP = 100
TOP_K = 3

EMBED_MODEL = "all-MiniLM-L6-v2"
GEN_MODEL = "google/flan-t5-base"


In [23]:
#load data
print("Loading dataset...")

with open(DATA_PATH, "r", encoding="utf-8", errors="ignore") as f:
    raw_text = f.read()


Loading dataset...


In [24]:
#generator
tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL)
generator = AutoModelForSeq2SeqLM.from_pretrained(GEN_MODEL)
print(GEN_MODEL)

google/flan-t5-base


In [25]:
def chunk_text(text, tokenizer, size=CHUNK_TOKENS, overlap=CHUNK_OVERLAP):

    # tokenize full text
    tokens = tokenizer.encode(text)

    chunks = []
    step = size - overlap

    for i in range(0, len(tokens), step):
        chunk_tokens = tokens[i:i+size]

        if len(chunk_tokens) < 200:  # avoid very small chunks
            continue

        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)

    return chunks

corpus = chunk_text(raw_text, tokenizer)

print("Total chunks:", len(corpus))

Token indices sequence length is longer than the specified maximum sequence length for this model (44582760 > 512). Running this sequence through the model will result in indexing errors


Total chunks: 148609


In [26]:
#embeddings
embedder = SentenceTransformer(EMBED_MODEL)

embeddings = embedder.encode(
    corpus,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
).astype("float32")

# normalize embeddings for cosine similarity
faiss.normalize_L2(embeddings)

# faiss index using inner product (cosine after normalization)
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)


Batches:   0%|          | 0/2323 [00:00<?, ?it/s]

In [27]:
#retrieval
def retrieve(question):

    q_emb = embedder.encode([question]).astype("float32")
    faiss.normalize_L2(q_emb)

    _, idx = index.search(q_emb, TOP_K)

    return [corpus[i] for i in idx[0]]



In [28]:
def truncate_context(chunks, max_tokens=1500):
  kept = []
  total = 0

  for c in chunks:
    tokens = tokenizer.encode(c)
    if total + len(tokens) > max_tokens:
      break
    kept.append(c)
    total += len(tokens)

  return kept

In [29]:
#post process
def post_process(text):

    # Remove parentheses
    text = re.sub(r"\([^)]*\)", "", text)

    # Remove citations like [1], [23]
    text = re.sub(r"\[\d+\]", "", text)

    # Remove extra whitespace
    text = re.sub(r"\s+", " ", text).strip()

    # Split sentences properly
    sentences = re.split(r'(?<=[.!?]) +', text)

    # Keep only 2–3 sentences
    sentences = [s.strip() for s in sentences if len(s.strip()) > 10]

    if len(sentences) >= 3:
        final = " ".join(sentences[:3])
    elif len(sentences) == 2:
        final = " ".join(sentences)
    elif len(sentences) == 1:
        final = sentences[0]
    else:
        final = text

    return final.strip()


In [30]:
#prompt builder
def build_prompt(question, context):

    return f"""
Answer using only the context.
Use Simple English.
Do not copy text exactly.
If the answer is not in the context, say exactly:
"Not enough information in the Simple Wikipedia dataset."

Context:
{context}

Question:
{question}

Answer:



"""


In [31]:
#pipeline
def answer(question):

    chunks = retrieve(question)
    chunks = truncate_context(chunks, max_tokens=500)
    context = "\n".join(chunks)

    prompt = build_prompt(question, context)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
        
    )

    outputs = generator.generate(
        **inputs,
        max_length=120,
        do_sample=False,
        temperature=0.0,
    )

    ans = tokenizer.decode(outputs[0], skip_special_tokens=True)
    ans = post_process(ans)

    return ans


In [33]:
#interactive loop
print("\nagent Ready")





agent Ready


In [35]:
while True:
    q = input("\nask question (type exit to stop): ")
    if q.lower() == "exit":
        break
    print("\nanswer:")
    print(answer(q))


ask question (type exit to stop):  exit


In [37]:
questions = [
    "What is the sun?",
    "What is water?",
    "What is photosynthesis?",
    "Who is the president of the United States of America?",
    "what is the population of mars?",
    "what is the capital of USA?"
]

for q in questions:
    print("Question:", q)
    print("Answer:", answer(q))
    print("-" * 60)


Question: What is the sun?
Answer: A star like many others in our Milky Way galaxy.
------------------------------------------------------------
Question: What is water?
Answer: a fluid.
------------------------------------------------------------
Question: What is photosynthesis?
Answer: Light-dependent reactions which need light to do work; and light-independent reactions which do not need light to do work.
------------------------------------------------------------
Question: Who is the president of the United States of America?
Answer: William McKinley Jr.
------------------------------------------------------------
Question: what is the population of mars?
Answer: Not enough information in the Simple Wikipedia dataset.
------------------------------------------------------------
Question: what is the capital of USA?
Answer: Washington, DC.
------------------------------------------------------------
